# Gradient descent variants

[`general-optimization`](general-optimization.ipynb) used `argmin` as a black box.
Here we open the box: **gradient descent by hand**, fitting the very
[linear regression](../02-regression/linear-regression.ipynb) model you already
know — so `.fit()` stops being magic.

Linear regression *has* a closed-form solution (the normal equations,
`(XᵀX)⁻¹Xᵀy`, which is likely what `smartcore` uses internally). Gradient descent
is the **iterative** alternative — and it's the one that matters when there's no
closed form (logistic regression's loss has none) or the data is too big to
invert a matrix. It's the workhorse behind almost all modern ML training.

In [ ]:
// All dependencies declared up front: adding a :dep in a later cell would make
// evcxr recompile and lose x/y (they're ndarray Array1 values).
:dep ndarray = { version = "0.15" }
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
:dep rayon = { version = "1" }
use ndarray::Array1;

// Fit y = w1*x + w0. Data: true line y = 2x + 1 with noise.
let n = 100usize;
let x: Array1<f64> = Array1::from((0..n).map(|i| i as f64 * 0.1).collect::<Vec<_>>());
let y: Array1<f64> = Array1::from((0..n).map(|i| 2.0 * (i as f64 * 0.1) + 1.0 + (((i * 7) % 5) as f64 - 2.0) * 0.3).collect::<Vec<_>>());

// Mean squared error, and its gradient over a chosen subset of rows.
fn mse(w0: f64, w1: f64, x: &Array1<f64>, y: &Array1<f64>) -> f64 {
    (0..x.len()).map(|i| (y[i] - (w0 + w1 * x[i])).powi(2)).sum::<f64>() / x.len() as f64
}
fn grad(w0: f64, w1: f64, x: &Array1<f64>, y: &Array1<f64>, idx: &[usize]) -> (f64, f64) {
    let m = idx.len() as f64;
    let (mut g0, mut g1) = (0.0, 0.0);
    for &i in idx { let r = y[i] - (w0 + w1 * x[i]); g0 -= 2.0 * r / m; g1 -= 2.0 * r * x[i] / m; }
    (g0, g1)   // dMSE/dw0, dMSE/dw1
}
// LCG shuffle for reproducible stochastic sampling (no rand dependency).
fn shuffled(n: usize, seed: u64) -> Vec<usize> {
    let mut v: Vec<usize> = (0..n).collect();
    let mut s = seed.wrapping_mul(6364136223846793005).wrapping_add(1);
    for i in (1..n).rev() { s = s.wrapping_mul(6364136223846793005).wrapping_add(1); v.swap(i, (s >> 33) as usize % (i + 1)); }
    v
}
println!("{} points; fitting y = w1*x + w0 (true: w1=2, w0=1)", n);

## Batch gradient descent

Use the gradient over the **entire** training set at each step. We plot the loss
curve (loss vs. iteration) and the **parameter path** — the trajectory of
`(w0, w1)` as it walks downhill toward the true `(1, 2)`:

In [ ]:
{
    use plotters::prelude::*;
    let all: Vec<usize> = (0..n).collect();
    let (mut w0, mut w1, lr, steps) = (0.0, 0.0, 0.02, 120);
    let mut loss = Vec::new();
    for _ in 0..steps {
        loss.push(mse(w0, w1, &x, &y));
        let (g0, g1) = grad(w0, w1, &x, &y, &all);
        w0 -= lr * g0; w1 -= lr * g1;
    }
    println!("converged to w0={:.3}, w1={:.3} (true 1, 2); final loss {:.4}", w0, w1, mse(w0, w1, &x, &y));
    evcxr_figure((460, 280), |root| {
        root.fill(&WHITE)?;
        let mut c = ChartBuilder::on(&root).caption("Batch GD: loss vs iteration", ("sans-serif", 15)).margin(8).x_label_area_size(30).y_label_area_size(45).build_cartesian_2d(0f64..steps as f64, 0f64..loss[0] * 1.05)?;
        c.configure_mesh().draw()?;
        c.draw_series(LineSeries::new((0..steps).map(|i| (i as f64, loss[i])), &BLUE))?;
        Ok(())
    })
}

In [ ]:
{
    use plotters::prelude::*;
    let all: Vec<usize> = (0..n).collect();
    let (mut w0, mut w1, lr, steps) = (0.0, 0.0, 0.02, 120);
    let mut path = Vec::new();
    for _ in 0..steps {
        path.push((w0, w1));
        let (g0, g1) = grad(w0, w1, &x, &y, &all);
        w0 -= lr * g0; w1 -= lr * g1;
    }
    // Parameter path: the trajectory of (w0, w1) toward the true (1, 2).
    evcxr_figure((460, 280), |root| {
        root.fill(&WHITE)?;
        let mut c = ChartBuilder::on(&root).caption("Parameter path (w0, w1) -> (1, 2)", ("sans-serif", 15)).margin(8).x_label_area_size(30).y_label_area_size(40).build_cartesian_2d(-0.2f64..1.6f64, -0.2f64..2.4f64)?;
        c.configure_mesh().x_desc("w0").y_desc("w1").draw()?;
        c.draw_series(LineSeries::new(path.iter().cloned(), &RED))?;
        c.draw_series(std::iter::once(Circle::new((1.0, 2.0), 4, GREEN.filled())))?;
        Ok(())
    })
}

## Stochastic & mini-batch

**SGD** uses a *single* random row per step — noisy but cheap. **Mini-batch**
uses a small random subset — the practical middle ground. Same start, same
learning rate; watch the loss curves. SGD/mini-batch are jumpier but each step
costs a fraction as much:

In [ ]:
{
    use plotters::prelude::*;
    let run = |batch: usize| -> Vec<f64> {
        let (mut w0, mut w1, lr, steps) = (0.0, 0.0, 0.02, 120);
        let mut loss = Vec::new();
        for step in 0..steps {
            loss.push(mse(w0, w1, &x, &y));
            let order = shuffled(n, step as u64);
            let idx = if batch >= n { &order[..] } else { &order[..batch] };
            let (g0, g1) = grad(w0, w1, &x, &y, idx);
            w0 -= lr * g0; w1 -= lr * g1;
        }
        loss
    };
    let (full, mini, sgd) = (run(n), run(16), run(1));
    evcxr_figure((480, 300), |root| {
        root.fill(&WHITE)?;
        let mut c = ChartBuilder::on(&root).caption("Loss: batch (blue) vs mini-16 (green) vs SGD (red)", ("sans-serif", 14)).margin(8).x_label_area_size(30).y_label_area_size(45).build_cartesian_2d(0f64..120f64, 0f64..full[0] * 1.05)?;
        c.configure_mesh().draw()?;
        c.draw_series(LineSeries::new((0..full.len()).map(|i| (i as f64, full[i])), &BLUE))?;
        c.draw_series(LineSeries::new((0..mini.len()).map(|i| (i as f64, mini[i])), &GREEN))?;
        c.draw_series(LineSeries::new((0..sgd.len()).map(|i| (i as f64, sgd[i])), &RED))?;
        Ok(())
    })
}

## Learning rate: the knob that makes or breaks it

Too small → crawls; just right → converges; too large → **oscillates or
diverges**. This is the single most common source of GD confusion, made visual:

In [ ]:
{
    use plotters::prelude::*;
    let all: Vec<usize> = (0..n).collect();
    let run_lr = |lr: f64| -> Vec<f64> {
        let (mut w0, mut w1) = (0.0, 0.0);
        (0..60).map(|_| { let l = mse(w0, w1, &x, &y); let (g0, g1) = grad(w0, w1, &x, &y, &all); w0 -= lr * g0; w1 -= lr * g1; l.min(20.0) }).collect()
    };
    let (slow, good, big) = (run_lr(0.003), run_lr(0.02), run_lr(0.16));
    evcxr_figure((480, 300), |root| {
        root.fill(&WHITE)?;
        let mut c = ChartBuilder::on(&root).caption("Learning rate: slow (blue) / good (green) / too big (red)", ("sans-serif", 13)).margin(8).x_label_area_size(30).y_label_area_size(45).build_cartesian_2d(0f64..60f64, 0f64..20f64)?;
        c.configure_mesh().draw()?;
        c.draw_series(LineSeries::new((0..slow.len()).map(|i| (i as f64, slow[i])), &BLUE))?;
        c.draw_series(LineSeries::new((0..good.len()).map(|i| (i as f64, good[i])), &GREEN))?;
        c.draw_series(LineSeries::new((0..big.len()).map(|i| (i as f64, big[i])), &RED))?;
        Ok(())
    })
}

## Momentum & Adam

**Momentum** adds an exponentially-weighted average of past gradients — it powers
through shallow spots and damps oscillation. **Adam** goes further with
per-parameter adaptive step sizes; it's *the* default optimizer across modern ML.
Both, from scratch, vs. plain gradient descent:

In [ ]:
{
    use plotters::prelude::*;
    let all: Vec<usize> = (0..n).collect();
    let steps = 80usize;
    let plain = { let (mut w0, mut w1) = (0.0, 0.0); (0..steps).map(|_| { let l = mse(w0,w1,&x,&y); let (g0,g1)=grad(w0,w1,&x,&y,&all); w0-=0.02*g0; w1-=0.02*g1; l }).collect::<Vec<f64>>() };
    let momentum = { let (mut w0, mut w1, mut v0, mut v1) = (0.0,0.0,0.0,0.0); (0..steps).map(|_| { let l = mse(w0,w1,&x,&y); let (g0,g1)=grad(w0,w1,&x,&y,&all); v0=0.9*v0+g0; v1=0.9*v1+g1; w0-=0.02*v0; w1-=0.02*v1; l }).collect::<Vec<f64>>() };
    let adam = {
        let (mut w0, mut w1) = (0.0, 0.0);
        let (mut m0, mut m1, mut s0, mut s1) = (0.0, 0.0, 0.0, 0.0);
        let (b1, b2, eps, lr) = (0.9_f64, 0.999_f64, 1e-8, 0.3);
        (1..=steps).map(|t| {
            let l = mse(w0, w1, &x, &y); let (g0, g1) = grad(w0, w1, &x, &y, &all);
            m0 = b1*m0 + (1.0-b1)*g0; m1 = b1*m1 + (1.0-b1)*g1;
            s0 = b2*s0 + (1.0-b2)*g0*g0; s1 = b2*s1 + (1.0-b2)*g1*g1;
            let (bc1, bc2) = (1.0 - b1.powi(t as i32), 1.0 - b2.powi(t as i32));
            let (mh0, mh1) = (m0/bc1, m1/bc1);
            let (sh0, sh1) = (s0/bc2, s1/bc2);
            w0 -= lr*mh0/(sh0.sqrt()+eps); w1 -= lr*mh1/(sh1.sqrt()+eps); l
        }).collect::<Vec<f64>>()
    };
    println!("final loss - plain {:.4}, momentum {:.4}, adam {:.4}", plain[steps-1], momentum[steps-1], adam[steps-1]);
    evcxr_figure((480, 300), |root| {
        root.fill(&WHITE)?;
        let mut c = ChartBuilder::on(&root).caption("plain (blue) / momentum (green) / Adam (red)", ("sans-serif", 14)).margin(8).x_label_area_size(30).y_label_area_size(45).build_cartesian_2d(0f64..steps as f64, 0f64..plain[0]*1.05)?;
        c.configure_mesh().draw()?;
        c.draw_series(LineSeries::new((0..steps).map(|i| (i as f64, plain[i])), &BLUE))?;
        c.draw_series(LineSeries::new((0..steps).map(|i| (i as f64, momentum[i])), &GREEN))?;
        c.draw_series(LineSeries::new((0..steps).map(|i| (i as f64, adam[i])), &RED))?;
        Ok(())
    })
}

## Parallelizing a mini-batch — does it pay off?

Computing per-row gradients within a batch is embarrassingly parallel, so you'd
reach for the [`rayon` pattern](../04b-multithreading/parallel-ml.ipynb). But
whether it *helps* depends on the work per row — watch what the benchmark
actually reports (kept in a block, since `rayon` at cell top level crashes
evcxr):

In [ ]:
{
    use rayon::prelude::*;
    use std::time::Instant;
    let (w0, w1) = (0.5, 1.5);
    let big_n = 2_000_000usize;   // pretend the batch is huge
    // sequential gradient sum
    let t = Instant::now();
    let seq: f64 = (0..big_n).map(|i| { let xi = (i % 100) as f64 * 0.1; -2.0 * ((2.0*xi+1.0) - (w0 + w1*xi)) * xi }).sum();
    let seq_t = t.elapsed();
    // parallel gradient sum
    let t = Instant::now();
    let par: f64 = (0..big_n).into_par_iter().map(|i| { let xi = (i % 100) as f64 * 0.1; -2.0 * ((2.0*xi+1.0) - (w0 + w1*xi)) * xi }).sum();
    let par_t = t.elapsed();
    println!("gradient over {} rows: sequential {:?}, parallel {:?} ({:.1}x)", big_n, seq_t, par_t, seq_t.as_secs_f64()/par_t.as_secs_f64());
    println!("same result: {}", (seq - par).abs() < 1e-3);
}

## Convergence, logistic regression, and what libraries really do

Notice the benchmark above often comes out **slower** in parallel: on this trivial
per-row work (one feature, two multiplications) thread overhead swamps the tiny
compute. You'd only win by parallelizing when each row is *expensive* (many
features) or the batch is enormous — exactly as the
[multithreading chapter](../04b-multithreading/parallel-ml.ipynb) showed with
heavier per-task work (a 7× speedup there). Parallelize deliberately, not reflexively.

**Stopping**: fixed iterations (used here) is simplest; better is to stop when the
loss stops improving, or to watch a validation score and stop early (the
[evaluation chapter's](../01d-evaluation/cross-validation.ipynb) split). Linear
and logistic loss surfaces are **convex**, so GD can't get stuck in a bad local
minimum — but that stops being true for non-convex models (neural nets), which is
exactly where these variants (momentum, Adam) earn their keep.

**Same mechanism, different loss**: [logistic regression](../02-regression/logistic-regression.ipynb)
has no closed form, so it's fit by this very procedure — swap MSE for cross-entropy
and its gradient. That chapter builds the full hand-rolled logistic version;
batch/mini-batch/SGD all carry over unchanged.

**What `.fit()` actually does**: `smartcore`/`linfa` use either a closed-form
solution or a more numerically careful, accelerated version of what we built here.
The point of this chapter is understanding the mechanism — not replacing the
library. For production general optimization, reach for `argmin`
([general-optimization](general-optimization.ipynb)).

Next: [hyperparameter search](hyperparameter-search.ipynb) — optimizing *across*
many fits, one level up from optimizing a single fit.